# UNSW-NB15 Dataset Analysis and Baseline Model

This notebook loads the UNSW-NB15 dataset, performs exploratory data analysis, preprocesses the data, and trains a baseline Random Forest classifier.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading

In [ ]:
train_path = r'../Datasets/USNW/UNSW_NB15_training-set.parquet'
test_path = r'../Datasets/USNW/UNSW_NB15_testing-set.parquet'

train_df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

## 2. Exploratory Data Analysis

In [ ]:
# Check distribution of the target variable
plt.figure(figsize=(6,4))
sns.countplot(data=train_df, x='label')
plt.title('Target Variable Distribution (0: Normal, 1: Attack)')
plt.show()

print(train_df['label'].value_counts())

In [ ]:
# Check for missing values
missing = train_df.isnull().sum()
print("Columns with missing values:")
print(missing[missing > 0])

## 3. Preprocessing
We will:
1. Drop the `id` column.
2. Drop the `attack_cat` column since we are doing binary classification on `label`.
3. Label encode categorical features (`proto`, `service`, `state`).
4. Standardize numerical features.

In [ ]:
# Drop unnecessary columns
if 'id' in train_df.columns:
    train_df = train_df.drop('id', axis=1)
if 'id' in test_df.columns:
    test_df = test_df.drop('id', axis=1)

if 'attack_cat' in train_df.columns:
    train_df = train_df.drop('attack_cat', axis=1)
if 'attack_cat' in test_df.columns:
    test_df = test_df.drop('attack_cat', axis=1)

X_train = train_df.drop('label', axis=1)
y_train = train_df['label']
X_test = test_df.drop('label', axis=1)
y_test = test_df['label']

In [ ]:
# Separate categorical and numerical columns
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

print("Categorical columns:", categorical_cols)

# Encode categoricals
for col in categorical_cols:
    le = LabelEncoder()
    unique_vals = list(set(X_train[col].astype(str).unique()) | set(X_test[col].astype(str).unique()))
    le.fit(unique_vals)
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

In [ ]:
# Scale numerical features
scaler = StandardScaler()
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

print("Preprocessing Complete. X_train shape:", X_train.shape)

## 4. Modeling (Baseline Random Forest)

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

## 5. Evaluation

In [ ]:
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()